In [1]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
load_dotenv()

True

In [2]:
model1 = ChatOllama(model="qwen2.5:3b")
model2 = ChatOllama(model="gemma3:1b")
template1 = PromptTemplate(
    template="Generate short and simple notes from the following text. \n {text}",
    input_variables=['text']
)
template2 = PromptTemplate(
    template="Generate 5 short question answers from the following text. \n {text}",
    input_variables=['text']
)
template3 = PromptTemplate(
    template="Merge the notes and quiz into a single document. \n notes -> {notes} and quiz -> {quiz}",
    input_variables=['notes','quiz']
)
parser = StrOutputParser()

In [4]:
from langchain_classic.schema.runnable import RunnableParallel
parallel_chain = RunnableParallel({
    'notes' : template1 | model1 | parser,
    'quiz' : template2 | model2 | parser
})
merge_chain = template3 | model1 | parser
chain = parallel_chain | merge_chain
# chain.get_graph().print_ascii()

In [6]:
text = """LLMs often generate text that is unstructured or inconsistent. Output parsers help convert this raw text into structured formats ensuring our application can reliably interpret and use the results.

Output parsers act as a bridge between the model and our application enforcing formats like JSON, lists or Python objects. This makes data extraction, validation and further processing seamless and consistent.
Structured Output Generation: Ensures model responses are formatted into JSON, lists or objects instead of plain text.
Schema Enforcement: Validates that the generated data follows a defined structure or schema before use.
Error Handling and Correction: Detects malformed outputs and can auto-correct them using tools like OutputFixingParser.
Multiple Parser Types: Supports different output formats such as String, List, JSON and Pydantic parsers for various use cases.
Integration with Chains: Works seamlessly with LangChain components like LLMChain or AgentExecutor to maintain consistent data flow.
Function
Some of the core functions of output parsers are:

Structure Data: Converts plain text into dictionaries, lists, JSON or custom objects. Example: "Name: John, Age: 30" to {"name": "John", "age": 30}.
Predictable Format: Defines expected fields and data types and helps downstream systems know exactly what to expect.
Reliable Processing: Simplifies database storage, API responses or further computations and reduces the need for additional text parsing.
Error Handling: Handles missing or extra fields gracefully and can validate and correct output before use.
Error Handling: Connects model output with your app’s workflows and makes integration with chains, agents or APIs seamless.
Integration in LangChain
Output Parsers in LangChain works in the following way:

LLM Generates Output: The model produces raw text in response to a prompt, which may be unstructured or inconsistent.
Parser Processes Text: The output parser converts this raw text into a structured format like JSON, lists or Python objects.
Application Uses Data: The structured data can be directly used in databases, APIs, UIs or further chains without extra processing.
Error Handling: Parsers detect missing fields, type mismatches or unexpected outputs and can correct or flag them.
Seamless Integration: Output parsers integrate with Chains, Agents and PromptTemplates to ensure consistent data flow throughout the workflow.
Types of Output Parsers
Types of Output Parsers are:

String Parsers: Convert LLM output directly into plain strings or simple text formats for easy use.
List Parsers: Split text into lists or arrays, useful for extracting multiple items from model output.
JSON or Dict Parsers: Parse LLM responses into JSON or Python dictionaries for structured data handling.
Pydantic Parsers: Use Pydantic models to enforce strict schemas and type validation on LLM outputs.
Output Fixing Parsers: Automatically correct or adjust outputs that don’t match the expected format.
Custom Parsers: User-defined parsers tailored for specific applications or complex output requirements."""

chain.invoke({'text':text})

'### Notes on Output Parsers\n\n- **LLMs generate unstructured text**, which needs to be **pulled into structured formats** via **output parsers**. This ensures application reliability in interpreting and processing the results.\n  \n- **Output parsers act as bridges**: They convert raw LLM output into JSON, lists or Python objects, facilitating seamless data extraction, validation, and further processing.\n\n- **Structured Output Generation**: Ensures that model responses are formatted into a specific structure like JSON, lists, or objects instead of plain text.\n  \n- **Schema Enforcement**: Validates generated data against defined structures to ensure consistency before use.\n  \n- **Error Handling and Correction**: Detects and corrects malformed outputs automatically using tools such as the `OutputFixingParser`.\n  \n- **Multiple Parser Types**: Supports different formats such as String, List, JSON or Pydantic parsers for diverse applications.\n\n- **Integration with Chains**: Seam

In [20]:
from langchain_classic.schema.runnable import RunnableBranch, RunnableLambda
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal
class Feedback(BaseModel):
    sentiment: Literal['Positive', 'Negative'] = Field(description="Give the sentiment of the feedback as positive or negative.")

parser2 = PydanticOutputParser(pydantic_object=Feedback)
prompt1 = PromptTemplate(
    template="Classify the sentiment of the following feedback text as positive or negative. {feedback} \n {format_instructions}",
    input_variables=['feedback'],
    partial_variables={'format_instructions': parser2.get_format_instructions()}
)
prompt2 = PromptTemplate(
    template="Write an appropriate response to this positive feedback, pretend you are the seller of the product discussed in the feedback.\n {feedback}",
    input_variables=['feedback']
)
prompt3 = PromptTemplate(
    template="Write an appropriate response to this negative feedback, pretend you are the seller of the product discussed in the feedback.\n {feedback}",
    input_variables=['feedback']
)

classifier_chain = prompt1 | model1 | parser2
condition_chain = RunnableBranch(
    (lambda x:x.sentiment == 'Positive', prompt2 | model1 | parser),
    (lambda x:x.sentiment == 'Negative', prompt3 | model1 | parser),
    RunnableLambda(lambda x: 'Could not find sentiment')
)
chain = classifier_chain | condition_chain
result = chain.invoke({'feedback': 'This is a useless phone not of any use.'})
result

"I apologize for any dissatisfaction with my product. I truly value your input and am committed to improving our offerings. Can you please share more details on what didn't meet your expectations? Your insights will help us ensure that we provide a better experience in the future."